<a href="https://colab.research.google.com/github/RobinJames25/task-app/blob/main/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
import json

next_task_id = 1
tasks = []
processed_requests = {}


def create_task(request_id, title, description):
    global next_task_id

    if not isinstance(request_id, str) or not isinstance(title, str) or not isinstance(description, str):
        return "Error: request_id, task title and description must be strings"

    if not request_id.strip() or not title.strip() or not description.strip():
        return "Error: request_id, task title and description required"

    if request_id not in processed_requests:
        task_id = f"TASK{next_task_id:03d}"

        task = {
            "request_id": request_id,
            "task_id": task_id,
            "title": title,
            "description": description,
            "status": "pending",
            "attempts": 0,
            "max_attempts": 3,
            "error_history": []
        }

        tasks.append(task)

        processed_requests[request_id] = {
            "title": title,
            "description": description
        }

        next_task_id += 1

        save_data(tasks, processed_requests, next_task_id)

    else:
        original_request = processed_requests[request_id]

        if (
            original_request["title"] == title
            and original_request["description"] == description
        ):
            return "Error: duplicate request ID"
        else:
            return "Error: request ID conflict"

    return task


def save_data(tasks, processed_requests, next_task_id):
    app_data = {
        "tasks": tasks,
        "processed_requests": processed_requests,
        "next_task_id": next_task_id
    }

    with open("tasks.json", "w") as file:
        json.dump(app_data, file)

def load_data():
    global tasks, processed_requests, next_task_id

    required_fields = [
        "request_id",
        "task_id",
        "title",
        "description",
        "status",
        "attempts",
        "max_attempts",
        "error_history"
    ]

    allowed_statuses = ["pending", "processing", "completed", "failed"]

    seen_task_ids = set()
    seen_request_ids = set()
    highest_task_id = 0

    try:
        with open("tasks.json", "r") as file:
            app_data = json.load(file)

            if not isinstance(app_data, dict):
                print("Error: Saved data must be a dictionary.")
                return False

            if (
                "tasks" not in app_data
                or "processed_requests" not in app_data
                or "next_task_id" not in app_data
            ):
                print("Error: Saved data is missing required fields.")
                return False

            if not (
                isinstance(app_data["tasks"], list)
                and isinstance(app_data["processed_requests"], dict)
                and type(app_data["next_task_id"]) is int
                and app_data["next_task_id"] > 0
            ):
                print("Error: Saved data contains invalid field types or values.")
                return False

            for task in app_data["tasks"]:
                if not isinstance(task, dict):
                    print("Error: Every task must be a dictionary.")
                    return False

            for task in app_data["tasks"]:

                for field in required_fields:
                    if field not in task:
                        print(f"Error: Task is missing required field: {field}")
                        return False

                if not isinstance(task["request_id"], str) or not task["request_id"].strip():
                    print("Error: request_id must be a non-empty string.")
                    return False

                if task["status"] not in allowed_statuses:
                    print(f"Error: Invalid task status: {task['status']}")
                    return False

                if type(task["attempts"]) is not int or task["attempts"] < 0:
                    print("Error: Attempts must be a non-negative integer.")
                    return False

                if (
                    type(task["max_attempts"]) is not int
                    or task["max_attempts"] > 3
                    or task["max_attempts"] <= 0
                ):
                    print("Error: max_attempts must be an integer between 1 and 3.")
                    return False
                if task["attempts"] > task["max_attempts"]:
                  print("Error: attempts cannot exceed max_attempts")
                  return False

                if not isinstance(task["error_history"], list):
                    print("Error: error_history must be a list.")
                    return False

                for error in task["error_history"]:
                    if not isinstance(error, str):
                        print("Error")
                        return False

                if not isinstance(task["title"], str) or not isinstance(task["description"], str):
                    print("Error: Task title and description must be strings.")
                    return False

                if not task["title"].strip() or not task["description"].strip():
                    print("Error: Task title and description cannot be empty.")
                    return False

                if task["status"] == "completed" and task["attempts"] < 1:
                    print("Error: A completed task must have at least one execution attempt.")
                    return False

                if task["status"] == "pending" and task["attempts"] >= task["max_attempts"]:
                    print("Error: Pending task has exhausted its maximum attempts.")
                    return False

                if not isinstance(task["task_id"], str):
                    print("Error: Task ID must be a string.")
                    return False

                if not (
                    task["task_id"].startswith("TASK")
                    and len(task["task_id"][4:]) == 3
                    and task["task_id"][4:].isdigit()
                ):
                    print("Error: Invalid task id.")
                    return False

                if int(task["task_id"][4:]) < 1:
                  print("Error: Task ID must be greater than TASK000.")
                  return False

                if task["task_id"] not in seen_task_ids:
                    seen_task_ids.add(task["task_id"])
                else:
                    print("Error: ID duplicate Error")
                    return False

                current_task_id = int(task["task_id"][4:])

                if current_task_id > highest_task_id:
                    highest_task_id = current_task_id

            for request_id, request_data in app_data["processed_requests"].items():
              if not isinstance(request_id, str) or not request_id.strip():
                print("Error: request_id must be a non-empty string")
                return False

              if not isinstance(request_data, dict):
                print("Error: request_data must be a dictionary.")
                return False

              if not "title" in request_data or not "description" in request_data:
                print("Error: request_data must contain title and description")
                return False

              if not isinstance(request_data["title"], str) or not isinstance(request_data["description"], str):
                print("Error: request_data title and description must be strings")
                return False

              if not request_data["title"].strip() or not request_data["description"].strip():
                print("Error: request data title and description must be non-empty string")
                return False

            for task in app_data["tasks"]:
              request_id = task["request_id"]

              if request_id in seen_request_ids:
                print("Error: Duplicate request ID.")
                return False

              seen_request_ids.add(request_id)

              if request_id not in app_data["processed_requests"]:
                print("Error: Task has no matching registry entry.")
                return False

              request_data = app_data["processed_requests"][request_id]

              if(
                  task["title"] != request_data["title"]
                  or task["description"] != request_data["description"]
              ):
                print("Error: Task and registry information do not match.")
                return False

            for request_id in app_data["processed_requests"]:
              if request_id not in seen_request_ids:
                print("Error: Registry entry has no matching task.")
                return False

            if app_data["next_task_id"] <= highest_task_id:
                print("Error: next_task_id must be greater than the highest existing task ID.")
                return False

            tasks = app_data["tasks"]
            processed_requests = app_data["processed_requests"]
            next_task_id = app_data["next_task_id"]

            return True

    except FileNotFoundError:
        tasks = []
        processed_requests = {}
        next_task_id = 1

        print("No saved data found. Starting a new application.")
        return True

    except json.JSONDecodeError:
        print("Error: Saved data is corrupted. Please investigate tasks.json.")
        return False
if __name__ == '__main__':

  result = load_data()

  print(result)
  new_task = create_task("REQ004", "Check stock", "Check available stock for a product")
  print(new_task)




True
{'request_id': 'REQ004', 'task_id': 'TASK004', 'title': 'Check stock', 'description': 'Check available stock for a product', 'status': 'pending', 'attempts': 0, 'max_attempts': 3, 'error_history': []}


In [55]:
print("Next task ID:", next_task_id)
print("Number of tasks:", len(tasks))
print("Task IDs:", [task["task_id"] for task in tasks])


Next task ID: 5
Number of tasks: 4
Task IDs: ['TASK001', 'TASK002', 'TASK003', 'TASK004']


In [56]:
with open("tasks.json", "r") as file:
    saved_data = json.load(file)

print("Saved next task ID:", saved_data["next_task_id"])
print("Saved task IDs:", [
    task["task_id"] for task in saved_data["tasks"]
])


Saved next task ID: 5
Saved task IDs: ['TASK001', 'TASK002', 'TASK003', 'TASK004']
